In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


In [2]:
# Load the dataset
df = pd.read_csv('final_internship_data.csv')
df.head()

,User ID,User Name,Driver Name,Car Condition,Weather,Traffic Condition,key,fare_amount,pickup_datetime,pickup_longitude,...,month,weekday,year,jfk_dist,ewr_dist,lga_dist,sol_dist,nyc_dist,distance,bearing
0,KHVrEVlD,Kimberly Adams,Amy Butler,Very Good,windy,Congested Traffic,2009-06-15 17:26:21.0000001,4.5,2009-06-15 17:26:21,-1.288826,...,6,0,2009,20.265840,55.176046,14.342611,34.543548,27.572573,1.030764,-2.918897
1,lPxIuEri,Justin Tapia,Hannah Zimmerman,Excellent,cloudy,Flow Traffic,2010-01-05 16:52:16.0000002,16.9,2010-01-05 16:52:16,-1.291824,...,1,1,2010,44.667679,31.832358,23.130775,15.125872,8.755732,8.450134,-0.375217
2,gsVN8JLS,Elizabeth Lopez,Amanda Jackson,Bad,stormy,Congested Traffic,2011-08-18 00:35:00.00000049,5.7,2011-08-18 00:35:00,-1.291242,...,8,3,2011,43.597686,33.712082,19.865289,17.722624,9.847344,1.389525,2.599961
3,9I7kWFgd,Steven Wilson,Amy Horn,Very Good,stormy,Flow Traffic,2012-04-21 04:30:42.0000001,7.7,2012-04-21 04:30:42,-1.291319,...,4,5,2012,42.642965,32.556289,21.063132,15.738963,7.703421,2.799270,0.133905
4,8QN5ZaGN,Alexander Andrews,Cassandra Larson,Bad,stormy,Congested Traffic,2010-03-09 07:51:00.000000135,5.3,2010-03-09 07:51:00,-1.290987,...,3,1,2010,43.329953,39.406828,15.219339,23.732406,15.600745,1.999157,-0.502703


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 26 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   User ID            500000 non-null  object 
 1   User Name          500000 non-null  object 
 2   Driver Name        500000 non-null  object 
 3   Car Condition      500000 non-null  object 
 4   Weather            500000 non-null  object 
 5   Traffic Condition  500000 non-null  object 
 6   key                500000 non-null  object 
 7   fare_amount        500000 non-null  float64
 8   pickup_datetime    500000 non-null  object 
 9   pickup_longitude   500000 non-null  float64
 10  pickup_latitude    500000 non-null  float64
 11  dropoff_longitude  499995 non-null  float64
 12  dropoff_latitude   499995 non-null  float64
 13  passenger_count    500000 non-null  int64  
 14  hour               500000 non-null  int64  
 15  day                500000 non-null  int64  
 16  mo

# Preprocssing

In [4]:
df.columns

Index(['User ID', 'User Name', 'Driver Name', 'Car Condition', 'Weather',
       'Traffic Condition', 'key', 'fare_amount', 'pickup_datetime',
       'pickup_longitude', 'pickup_latitude', 'dropoff_longitude',
       'dropoff_latitude', 'passenger_count', 'hour', 'day', 'month',
       'weekday', 'year', 'jfk_dist', 'ewr_dist', 'lga_dist', 'sol_dist',
       'nyc_dist', 'distance', 'bearing'],
      dtype='object')

In [5]:
# Clean

# Convert 'pickup_datetime' to datetime type
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"],errors='coerce')

# make all th col lower case and repalce space with _
df.columns = df.columns.str.replace(' ','_',regex=True)
df.columns = df.columns.str.lower()

# Extract new time-based features
df["pickup_hour"] = df["pickup_datetime"].dt.strftime("%I %p")  # 12-hour format with AM/PM
df["pickup_day"] = df["pickup_datetime"].dt.strftime("%A")  # Full weekday name (e.g., Monday)
df["pickup_month"] = df["pickup_datetime"].dt.strftime("%B")  # Full month name (e.g., March)
df["pickup_year"] = df["pickup_datetime"].dt.strftime("%B %Y")  # Month + Year (e.g., March 2024)

# Drop the original 'pickup_datetime' column since we extracted useful features
df = df.drop(columns=["pickup_datetime"])

In [6]:
# drop unnecessary cols
df= df.drop(columns=['user_id', 'key'])

# Drop redundant time-based columns
columns_to_remove = ["hour", "day", "month", "year"]
# Remove selected columns
df = df.drop(columns=columns_to_remove)

In [7]:
#fill Na for numric col
df = df.apply(lambda col: col.fillna(col.median()) if col.dtype in ['float64', 'int64'] else col, axis=0)

In [8]:
# split our data to numric and category
num = df.select_dtypes('number')
cat = df.select_dtypes('object')

In [9]:
num.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 14 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   fare_amount        500000 non-null  float64
 1   pickup_longitude   500000 non-null  float64
 2   pickup_latitude    500000 non-null  float64
 3   dropoff_longitude  500000 non-null  float64
 4   dropoff_latitude   500000 non-null  float64
 5   passenger_count    500000 non-null  int64  
 6   weekday            500000 non-null  int64  
 7   jfk_dist           500000 non-null  float64
 8   ewr_dist           500000 non-null  float64
 9   lga_dist           500000 non-null  float64
 10  sol_dist           500000 non-null  float64
 11  nyc_dist           500000 non-null  float64
 12  distance           500000 non-null  float64
 13  bearing            500000 non-null  float64
dtypes: float64(12), int64(2)
memory usage: 53.4 MB


In [10]:
cat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   user_name          500000 non-null  object
 1   driver_name        500000 non-null  object
 2   car_condition      500000 non-null  object
 3   weather            500000 non-null  object
 4   traffic_condition  500000 non-null  object
 5   pickup_hour        500000 non-null  object
 6   pickup_day         500000 non-null  object
 7   pickup_month       500000 non-null  object
 8   pickup_year        500000 non-null  object
dtypes: object(9)
memory usage: 34.3+ MB


In [11]:
def remove_outliers(df, num_cols):
    """
    Removes outliers using the IQR method for multiple numerical columns.
    
    Parameters:
    df (DataFrame): The dataset
    num_cols (list): List of numerical column names
    
    Returns:
    DataFrame: Cleaned dataset without outliers
    """
    for col in num_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    
    return df


In [12]:
# Remove outliers from key numerical columns
df = remove_outliers(df,num)

# Check new shape after outlier removal
df.columns

Index(['user_name', 'driver_name', 'car_condition', 'weather',
       'traffic_condition', 'fare_amount', 'pickup_longitude',
       'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude',
       'passenger_count', 'weekday', 'jfk_dist', 'ewr_dist', 'lga_dist',
       'sol_dist', 'nyc_dist', 'distance', 'bearing', 'pickup_hour',
       'pickup_day', 'pickup_month', 'pickup_year'],
      dtype='object')

# Modeling 

In [13]:
# Re-import necessary libraries
import optuna
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [14]:
# Encoding categorical variables
label_encoders = {}
for col in cat:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

In [15]:
df.head()

,user_name,driver_name,car_condition,weather,traffic_condition,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,...,ewr_dist,lga_dist,sol_dist,nyc_dist,distance,bearing,pickup_hour,pickup_day,pickup_month,pickup_year
2,55025,6232,0,2,0,5.7,-1.291242,0.711418,-1.291391,0.711231,...,33.712082,19.865289,17.722624,9.847344,1.389525,2.599961,22,4,1,9
3,156412,7543,3,2,2,7.7,-1.291319,0.710927,-1.291396,0.711363,...,32.556289,21.063132,15.738963,7.703421,2.799270,0.133905,6,2,0,3
4,2865,28191,0,2,0,5.3,-1.290987,0.711536,-1.290787,0.711811,...,39.406828,15.219339,23.732406,15.600745,1.999157,-0.502703,12,5,7,47
5,92990,170827,1,3,0,12.1,-1.291560,0.710901,-1.291071,0.711365,...,32.824930,20.648176,15.741133,7.649541,3.787239,-0.674251,16,4,4,28
6,69847,27803,1,4,2,7.5,-1.291195,0.711251,-1.291086,0.711481,...,35.482608,18.113693,19.126670,10.993150,1.555807,-0.342260,15,5,9,63


In [16]:
# Define features and target
X = df.drop(columns=["fare_amount"])  # Features
y = df["fare_amount"]  # Target variable

In [17]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
# Normalize numerical features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Modeling

In [21]:
# Define models to optimize
models = {
    "LinearRegression": LinearRegression(),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42)
}

# Define the Optuna objective function
def objective(trial):
    model_name = trial.suggest_categorical("model", list(models.keys()))  # Select a model
    if model_name == "LinearRegression":
        model = LinearRegression()

    elif model_name == "DecisionTree":
        model = DecisionTreeRegressor(
            max_depth=trial.suggest_int("max_depth", 3, 15),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 10),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 7),
            random_state=42
        )

    elif model_name == "RandomForest":
        model = RandomForestRegressor(
            n_estimators=trial.suggest_int("n_estimators", 50, 200, step=50),
            max_depth=trial.suggest_int("max_depth", 5, 15),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 10),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 7),
            random_state=42,
            n_jobs=-1
        )

     #  Create pipeline
    pipeline = Pipeline([
        ("model", model)  # Chosen regression model
    ])

    # Perform cross-validation
    scores = cross_val_score(pipeline, X_train, y_train, scoring="neg_root_mean_squared_error", cv=3)
    return np.mean(scores)  # Optuna maximizes score, so we use negative RMSE


In [22]:
#  Run Optuna optimization
study = optuna.create_study(direction="maximize")  # Higher score (lower RMSE) is better
study.optimize(objective, n_trials=15)  # Run 10 trials

[I 2025-03-08 14:25:59,354] A new study created in memory with name: no-name-b5124455-d37c-4a8e-a822-789287212921
[I 2025-03-08 14:26:07,678] Trial 0 finished with value: -2.2193645671030713 and parameters: {'model': 'DecisionTree', 'max_depth': 3, 'min_samples_split': 7, 'min_samples_leaf': 5}. Best is trial 0 with value: -2.2193645671030713.
[I 2025-03-08 14:26:21,603] Trial 1 finished with value: -2.1179083458259917 and parameters: {'model': 'DecisionTree', 'max_depth': 6, 'min_samples_split': 2, 'min_samples_leaf': 4}. Best is trial 1 with value: -2.1179083458259917.
[I 2025-03-08 14:34:28,566] Trial 2 finished with value: -2.0000429475286063 and parameters: {'model': 'RandomForest', 'n_estimators': 200, 'max_depth': 11, 'min_samples_split': 3, 'min_samples_leaf': 6}. Best is trial 2 with value: -2.0000429475286063.
[I 2025-03-08 14:34:38,280] Trial 3 finished with value: -2.152933727844792 and parameters: {'model': 'DecisionTree', 'max_depth': 5, 'min_samples_split': 3, 'min_sampl

In [24]:
#  Train the best model
best_params = study.best_params
best_model_name = best_params.pop("model")
best_model = models[best_model_name]

# If the best model is not LinearRegression, apply the best hyperparameters
if best_model_name != "LinearRegression":
    best_model.set_params(**best_params)

#  Final pipeline with best model
final_pipeline = Pipeline([
    ("model", best_model)
])

#  Train & test best model
final_pipeline.fit(X_train, y_train)
y_pred = final_pipeline.predict(X_test)

#  Evaluate performance
rmse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Best Model: {best_model_name}")
print(f"Best Hyperparameters: {best_params}")
print(f"Test RMSE: {rmse:.4f}")
print(f"R² Score: {r2}")

Best Model: RandomForest
Best Hyperparameters: {'n_estimators': 100, 'max_depth': 14, 'min_samples_split': 3, 'min_samples_leaf': 2}
Test RMSE: 3.9420


# Other Modeling 

In [27]:
import numpy as np
import xgboost as xgb
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR, LinearSVR,NuSVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, ShuffleSplit
from sklearn.metrics import mean_squared_error, r2_score

In [29]:
# Define Models
models = {
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, objective="reg:squarederror", random_state=42, n_jobs=-1),
    "SVR": LinearSVR(epsilon=0.1)
}

# Train & Evaluate Each Model
results = {}
for name, model in models.items():
    print(f"\n🔍 Training {name}...")

    # Train the model
    model.fit(X_train, y_train)

    # Predict on test data
    y_pred = model.predict(X_test)

    # Compute Metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # Store results
    results[name] = {"RMSE": rmse, "R² Score": r2}

    print(f"   ✅ RMSE: {rmse:.4f}")
    print(f"   ✅ R² Score: {r2:.4f}")


🔍 Training GradientBoosting...
   ✅ RMSE: 1.8336
   ✅ R² Score: 0.7211

🔍 Training XGBoost...
   ✅ RMSE: 1.8389
   ✅ R² Score: 0.7195

🔍 Training SVR...
   ✅ RMSE: 2.1314
   ✅ R² Score: 0.6232


In [30]:
# Find the Best Model (Based on RMSE)
best_model = min(results, key=lambda x: results[x]["RMSE"])

print("\n🚀 Best Model:")
print(f"   🔥 Model: {best_model}")
print(f"   ✅ RMSE: {results[best_model]['RMSE']:.4f}")
print(f"   ✅ R² Score: {results[best_model]['R² Score']:.4f}")


🚀 Best Model:
   🔥 Model: GradientBoosting
   ✅ RMSE: 1.8336
   ✅ R² Score: 0.7211
